# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIRˆ² colorectal cancer survivor dataset using the [`mlcroissant`](https://mlcroissant.org) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if not yet installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
md = dataset.metadata
print(f"{md.name}: {md.description}\n")
print(f"Identifier: {md.identifier}")
print(f"Version: {md.version}")
print(f"Published: {md.datePublished}")
print(f"License: {md.license}")

## 2. Data Overview
Review available record sets and their fields (with their `@id`s). These enable granular data access and help clarify how to reference each part of the dataset. All Croissant entities are referenced by their `@id`.

In [ ]:
# List all record sets with @id and involved fields
print("Record sets available in the dataset:")
record_sets = list(dataset.record_sets)
for record_set in record_sets:
    print(f"- Record set @id: {record_set.id}")
    print(f"  name: {getattr(record_set, 'name', '')}")
    if hasattr(record_set, 'fields'):
        print("  Fields (with @id):")
        for field in record_set.fields:
            print(f"    - {field.id} : {getattr(field, 'name', '')}")
    print()

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. Reference record sets and fields by their `@id`.

In [ ]:
# Extract data from each record set and load into a dictionary of DataFrames.
rs_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}
print(f"Loading records for each record set: {rs_ids}\n")

for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"{rs_id} columns: {df.columns.tolist()}")
    print(df.head(2), end="\n\n")

# Select one record set for further analysis (update the value if multiple are available)
if rs_ids:
    chosen_record_set_id = rs_ids[0]
    chosen_df = dataframes[chosen_record_set_id]
else:
    chosen_record_set_id = None
    chosen_df = None

## 4. Exploratory Data Analysis (EDA)
Perform basic EDA: filter records, normalize numeric variables, group by key field(s). When referencing fields, always use their Croissant `@id` directly as the column name.

In [ ]:
import numpy as np

if chosen_df is not None and not chosen_df.empty:
    print(f"Preview: {chosen_record_set_id}")
    display(chosen_df.head())
    
    # Try to identify numeric fields by dtype or heuristic
    candidate_numeric_fields = [col for col in chosen_df.columns if np.issubdtype(chosen_df[col].dropna().apply(type).mode()[0], np.number)]
    if not candidate_numeric_fields:
        candidate_numeric_fields = [col for col in chosen_df.columns if 'age' in col.lower() or 'count' in col.lower()]

    print(f"Numeric field candidates: {candidate_numeric_fields}")
    
    if candidate_numeric_fields:
        numeric_field_id = candidate_numeric_fields[0]
        # Use a threshold at 75% quantile for demonstration
        if pd.api.types.is_numeric_dtype(chosen_df[numeric_field_id]):
            threshold = chosen_df[numeric_field_id].quantile(0.75)
        else:
            # Attempt to coerce to numeric
            chosen_df[numeric_field_id] = pd.to_numeric(chosen_df[numeric_field_id], errors='coerce')
            threshold = chosen_df[numeric_field_id].quantile(0.75)

        filtered_df = chosen_df[chosen_df[numeric_field_id] > threshold].copy()
        print(f"Filtered rows where {numeric_field_id} > {threshold} (using @id as column name):")
        display(filtered_df.head())

        # Normalize the numeric field
        col_norm = f"{numeric_field_id}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, col_norm]].head())
    else:
        filtered_df = chosen_df
        print("No obvious numeric fields to filter or normalize.")
        col_norm = None

    # Try to identify a likely grouping field (@id)
    candidate_group_fields = [col for col in chosen_df.columns if 'sex' in col.lower() or 'msi' in col.lower() or 'group' in col.lower() or 'anatom' in col.lower() or 'site' in col.lower() or 'label' in col.lower()]
    if candidate_group_fields:
        group_field = candidate_group_fields[0]
        if group_field in filtered_df.columns and col_norm:
            grouped_df = filtered_df.groupby(group_field)[[numeric_field_id, col_norm]].mean(numeric_only=True)
            print(f"Grouped by {group_field} (mean of numeric field and normalized):")
            display(grouped_df.head())
    else:
        print("No grouping field (categorical @id) detected.")

## 5. Visualization
Visualize distributions or relationships between fields using the DataFrame extracted from a record set.

> **Note**: If the dataset has fewer than 2 numeric columns, a bar plot or value count plot of a categorical field will be shown instead.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_df is not None and not chosen_df.empty:
    # Visualize numeric field distribution
    num_cols = [col for col in chosen_df.columns if pd.api.types.is_numeric_dtype(chosen_df[col]) or chosen_df[col].dropna().apply(lambda v: str(v).replace('.', '', 1).isdigit()).any()]

    if num_cols:
        col = num_cols[0]
        sns.histplot(pd.to_numeric(chosen_df[col], errors='coerce').dropna(), kde=True)
        plt.title(f"Distribution of {col} (@id)")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.show()

    # If at least two numeric columns, show a scatter/correlation
    if len(num_cols) > 1:
        plt.figure(figsize=(6, 4))
        sns.scatterplot(x=chosen_df[num_cols[0]], y=chosen_df[num_cols[1]])
        plt.title(f"Scatter: {num_cols[0]} vs {num_cols[1]}")
        plt.xlabel(num_cols[0])
        plt.ylabel(num_cols[1])
        plt.show()
    else:
        # Show a barplot for a likely categorical
        cat_cols = [col for col in chosen_df.columns if chosen_df[col].dtype == object or chosen_df[col].dtype.name == 'category']
        if cat_cols:
            col = cat_cols[0]
            plt.figure(figsize=(7, 4))
            chosen_df[col].value_counts().plot(kind='bar')
            plt.title(f"Value counts for {col} (@id)")
            plt.ylabel("Count")
            plt.xlabel(col)
            plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load Croissant metadata and records using the `mlcroissant` library, referencing all entities by their `@id`.
- Inspect available record sets, field ids, and corresponding data.
- Extract dataset tables as DataFrames and perform exploratory data analysis.
- Filter, normalize, group, and visualize data.

The FAIRˆ² colorectal dataset is now ready for further modeling or research questions. For robust referencing in downstream analysis, keep using exact Croissant `@id` fields!